# Krycklan GLUE results analysis

Saved outputs may be from an earlier analysis. Rerun the cells to refresh them. The time-series section reruns every selected simulation and can be computationally expensive.

## Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv()
sns.set_context('notebook', font_scale=1.5)

#%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
results_folder = os.environ.get('SPAFHY_RESULTS_FOLDER')
if not results_folder or not results_folder.strip():
    raise EnvironmentError('SPAFHY_RESULTS_FOLDER is not set. Define it in .env or the environment.')
#glue_results_csv = Path(results_folder).expanduser() / 'krycklan' / 'glue_krycklan' / '20261105131819' / 'glue_results.csv'
#glue_results_csv = Path(results_folder).expanduser() / 'krycklan' / 'glue_krycklan' / '20261106133122' / 'glue_results.csv'
#glue_results_csv = Path(results_folder).expanduser() / 'krycklan' / 'glue_krycklan' / '20261109150350' / 'glue_results.csv'
glue_results_csv = Path(results_folder).expanduser() / 'krycklan' / 'glue_krycklan' / '20m' / '20261118165227' / 'glue_results.csv'

soil_type = 'Till'
base_metrics = ['kge_tot', 'rmse_tot', 'mbe_tot', 'kge_sbsrf', 'rmse_sbsrf', 'mbe_sbsrf']
parameter_columns = [f'kmax_{soil_type}', f'kmin_{soil_type}', f'f_{soil_type}']

data = pd.read_csv(glue_results_csv)
required_columns = ['sample_id', *parameter_columns, *base_metrics]
missing = set(required_columns) - set(data.columns)
if missing:
    raise ValueError(f'Missing columns in {glue_results_csv}: {sorted(missing)}')
if data.empty:
    raise ValueError(f'No samples in {glue_results_csv}')
if data['sample_id'].isna().any() or data['sample_id'].duplicated().any():
    raise ValueError('sample_id values must be non-null and unique')
data[parameter_columns + base_metrics] = data[parameter_columns + base_metrics].apply(pd.to_numeric, errors='raise')
if not np.isfinite(data[parameter_columns + base_metrics].to_numpy()).all():
    raise ValueError('Parameter and metric values must be finite')
if (data[[f'kmax_{soil_type}', f'kmin_{soil_type}']] <= 0).any().any():
    raise ValueError('kmax and kmin must be positive')
for metric in ('kge', 'rmse', 'mbe'):
    data[f'{metric}_ave'] = data[f'{metric}_tot'] / 2 + data[f'{metric}_sbsrf'] / 2
metrics = base_metrics + [f'{metric}_ave' for metric in ('kge', 'rmse', 'mbe')]
data = data.set_index('sample_id', drop=False)
print(f'Loaded {len(data)} samples from {glue_results_csv}')
print(f'Recorded kmin values: {data[f"kmin_{soil_type}"].unique()}')

# Calculate rank for each sample_id and metric

In [41]:
for metric_name in metrics:
    values = data[metric_name].abs() if metric_name.startswith('mbe_') else data[metric_name]
    ascending = not metric_name.startswith('kge_')
    data[f'rank_{metric_name}'] = values.rank(method='min', ascending=ascending)

In [ ]:
# Adjust these weights to change the importance of each metric in the final ranking.
w_kge = 1.0
w_rmse = 0.
w_mbe = 0.
N = len(data)
for suffix in ('tot', 'sbsrf', 'ave'):
    data[f'points_{suffix}'] = (
        w_kge * (N - data[f'rank_kge_{suffix}'])
        + w_rmse * (N - data[f'rank_rmse_{suffix}'])
        + w_mbe * (N - data[f'rank_mbe_{suffix}'])
    )
    data[f'rank_{suffix}'] = data[f'points_{suffix}'].rank(method='min', ascending=False)

# GLUE analysis

The average case uses the arithmetic mean of the total and subsurface KGE,
RMSE, and signed MBE for each sample; it is not a metric of average runoff.
Choose a simulation count or KGE cutoff separately for each case. Set a
cutoff to a number to select all simulations with KGE at or
above it; the corresponding count is ignored. With a cutoff of `None`, the
count selects the highest-KGE simulations, resolving ties by CSV order. Set
a count to zero with no cutoff to skip that component; at least one component
must be selected. `color_metric` controls the parameter-plot colors.


In [ ]:
from numbers import Real

color_metric = 'kge'  # kge, rmse, mbe, or points; used for plot colors
n_sims_tot = 1
n_sims_sbsrf = 0
n_sims_ave = 0
kge_cutoff_tot = None  # Set a number to select by KGE instead of count
kge_cutoff_sbsrf = None
kge_cutoff_ave = None
if color_metric not in ('kge', 'rmse', 'mbe', 'points'):
    raise ValueError(f'Unsupported color metric: {color_metric}')


def select_simulations(data, suffix, count, cutoff):
    column = f'kge_{suffix}'
    if cutoff is not None:
        if isinstance(cutoff, bool) or not isinstance(cutoff, Real) or not np.isfinite(cutoff):
            raise ValueError(f'kge_cutoff_{suffix} must be a finite number or None')
        selected = data.loc[data[column] >= cutoff].sort_values(column, ascending=False, kind='stable')
        if selected.empty:
            raise ValueError(f'No {suffix} simulations meet kge_cutoff_{suffix}={cutoff}')
        return selected, cutoff

    if not isinstance(count, int) or isinstance(count, bool) or count < 0:
        raise ValueError(f'n_sims_{suffix} must be a non-negative integer')
    if count > len(data):
        raise ValueError(f'n_sims_{suffix}={count} exceeds the {len(data)} available samples')
    selected = data.nlargest(count, column, keep='first')
    return selected, selected[column].iloc[-1] if count else None


suitable_simulations_tot, threshold_tot = select_simulations(data, 'tot', n_sims_tot, kge_cutoff_tot)
suitable_simulations_sbsrf, threshold_sbsrf = select_simulations(data, 'sbsrf', n_sims_sbsrf, kge_cutoff_sbsrf)
suitable_simulations_ave, threshold_ave = select_simulations(data, 'ave', n_sims_ave, kge_cutoff_ave)
n_sims_tot = len(suitable_simulations_tot)
n_sims_sbsrf = len(suitable_simulations_sbsrf)
n_sims_ave = len(suitable_simulations_ave)
if n_sims_tot == 0 and n_sims_sbsrf == 0 and n_sims_ave == 0:
    raise ValueError('At least one of n_sims_tot, n_sims_sbsrf and n_sims_ave must be positive')



In [ ]:
def metric_values(frame, metric_name, suffix):
    values = frame[f'{metric_name}_{suffix}']
    return values.abs() if metric_name == 'mbe' else values


fig, ax = plt.subplots(1, 3, figsize=(20, 6))
fig.subplots_adjust(wspace=0.3)
for idx, (suffix, cutoff, num_sims) in enumerate((
    ('tot', threshold_tot, n_sims_tot),
    ('sbsrf', threshold_sbsrf, n_sims_sbsrf),
    ('ave', threshold_ave, n_sims_ave),
)):
    values = data[f'kge_{suffix}']
    sns.histplot(x=values, bins=100, kde=True, ax=ax[idx])
    if cutoff is not None:
        ax[idx].axvline(cutoff, color='red', linestyle='--', label=f'KGE threshold: {cutoff}')
    ax[idx].set_xlabel(f'kge_{suffix}')
    ax[idx].set_title(f'{suffix.upper()} KGE ({num_sims} simulations selected)')
    if cutoff is not None:
        ax[idx].legend()

In [ ]:
for label, suitable, cutoff in (
    ('Total runoff', suitable_simulations_tot, threshold_tot),
    ('Subsurface runoff', suitable_simulations_sbsrf, threshold_sbsrf),
    ('Average metrics', suitable_simulations_ave, threshold_ave),
):
    if cutoff is None:
        print(f'{label}: skipped (0 simulations selected)')
    else:
        print(f'{label}: selected {len(suitable)} simulations; KGE threshold={cutoff}')

## Visualize sampled kmax and f

The top row shows all simulations over the configured full ranges, extended
if necessary to include the selected simulations' axis extent. The bottom
row shows selected simulations; skipped components have no bottom-row plot.
Each row shares a color scale across its panels. The top colorbar uses
tenths and the bottom colorbar uses hundredths.
The current GLUE runner fixes `kmin`; historical CSVs may contain varying
`kmin` values.


In [27]:
print(f'kmax range in CSV: {data[f"kmax_{soil_type}"].min():.3g}–{data[f"kmax_{soil_type}"].max():.3g} m/s')
print(f'f range in CSV: {data[f"f_{soil_type}"].min():.3g}–{data[f"f_{soil_type}"].max():.3g}')

In [1]:
from matplotlib.ticker import MaxNLocator

F_BOUNDS = (2.0, 20.0)
KMAX_BOUNDS = (5e-4, 0.01)

fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharex='row', sharey='row')
fig.subplots_adjust(wspace=0.1, hspace=0.25)

suitable_sets = [suitable_simulations_tot, suitable_simulations_sbsrf, suitable_simulations_ave]
suffixes = ['tot', 'sbsrf', 'ave']
case_titles = ['Total', 'Subsurface', 'Average']
for row in (1, 0):
    row_axes = axes[row]
    full_range = row == 0
    decimals = 1 if full_range else 2
    scale = 10 ** decimals
    row_data = [data] * 3 if full_range else suitable_sets
    row_colors = [metric_values(frame, color_metric, suffix)
                  for frame, suffix in zip(row_data, suffixes) if not frame.empty]
    color_min = np.floor(min(values.min() for values in row_colors) * scale) / scale
    color_max = np.ceil(max(values.max() for values in row_colors) * scale) / scale
    if color_min == color_max:
        color_max = color_min + 1 / scale
    y_label_shown = False
    for col, (ax, suitable, suffix, case_title) in enumerate(zip(
        row_axes, suitable_sets, suffixes, case_titles
    )):
        ax.text(0.9, 0.97, f'{chr(ord("A") + row * 3 + col)})',
                transform=ax.transAxes, ha='left', va='top')
        if full_range:
            ax.set_title(case_title)
        if not full_range and suitable.empty:
            ax.set_axis_off()
            ax.set_title(f'{suffix.upper()} skipped (0 simulations selected)')
            continue
        plot_data = data if full_range else suitable
        colors = metric_values(plot_data, color_metric, suffix)
        scatter = ax.scatter(
            plot_data[f'kmax_{soil_type}'], plot_data[f'f_{soil_type}'],
            c=colors, cmap='viridis', vmin=color_min, vmax=color_max,
            s=10 if full_range else 10,
            alpha=0.6 if full_range else 0.6
        )
        ax.set_xscale('log')
        if full_range:
            f_padding = (F_BOUNDS[1] - F_BOUNDS[0]) * 0.05
            selected_xlim = axes[1, 0].get_xlim()
            selected_ylim = axes[1, 0].get_ylim()
            ax.set_xlim(
                min(KMAX_BOUNDS[0] / 10**0.05, selected_xlim[0]),
                max(KMAX_BOUNDS[1] * 10**0.05, selected_xlim[1]),
            )
            ax.set_ylim(
                min(F_BOUNDS[0] - f_padding, selected_ylim[0]),
                max(F_BOUNDS[1] + f_padding, selected_ylim[1]),
            )
            #if not suitable.empty:
                #ax.plot(
                    #[selected_xlim[0], selected_xlim[1], selected_xlim[1], selected_xlim[0], selected_xlim[0]],
                    #[selected_ylim[0], selected_ylim[0], selected_ylim[1], selected_ylim[1], selected_ylim[0]],
                    #color='red', linestyle='--', linewidth=1.5, label='Selected-row extent'
                #)
                #ax.legend(loc='upper right')
            #ax.set_title(f'{suffix.upper()} all simulations' + (' with suitable extent' if not suitable.empty else ''))
        #else:
        #    ax.set_title(f'{suffix.upper()} suitable simulations (n={len(suitable)})')

        if row == 1:
            ax.set_xlabel(r'$K_{h,\mathrm{sat},\mathrm{max}}$ [m s$^{-1}$]')
        if not y_label_shown:
            ax.set_ylabel(r'$f$ [m$^{-1}$]')
            ax.tick_params(axis='y', labelleft=True)
            y_label_shown = True
    ticks = MaxNLocator(nbins=5, integer=True).tick_values(color_min * scale, color_max * scale) / scale
    ticks = ticks[(ticks >= color_min - 1e-10) & (ticks <= color_max + 1e-10)]
    color_label = 'abs(MBE)' if color_metric == 'mbe' else color_metric.upper()
    fig.colorbar(scatter, ax=row_axes.tolist(), label=color_label, ticks=ticks, format=f'%.{decimals}f')

NameError: name 'plt' is not defined

## Visualize Ksat curves of suitable simulations

Using the current soil-generator defaults, conductivity is `kmax` down to
the constant surface depth, then decays as

$$K_{sat} = (K_{max} - K_{min})e^{-f(d-d_c)} + K_{min}, \quad d > d_c.$$

Here $d$ is positive depth and $d_c$ is the constant surface depth. Historical
results may have used different soil-generator settings.

In [ ]:
from scripts.create_soil_params import _CONST_SURF_KMAX, _MAX_DEPTH

constant_depth = _CONST_SURF_KMAX[soil_type]
depth = np.linspace(0, _MAX_DEPTH[soil_type], 100)
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for ax, suitable, title in zip(
    axes,
    [suitable_simulations_tot, suitable_simulations_sbsrf, suitable_simulations_ave],
    ['TOT suitable simulations', 'SBSRF suitable simulations', 'AVE suitable simulations']
):
    if suitable.empty:
        ax.set_axis_off()
        ax.set_title(f'{title} (skipped)')
        continue
    for _, row in suitable.iterrows():
        kmin = row[f'kmin_{soil_type}']
        kmax = row[f'kmax_{soil_type}']
        f = row[f'f_{soil_type}']
        below_const = np.maximum(depth - constant_depth, 0)
        k_sat = (kmax - kmin) * np.exp(-f * below_const) + kmin
        ax.plot(np.log10(k_sat), -depth, color='grey', alpha=0.1)
    ax.set_xlabel('log10(k$_{sat}$ [m/s])')
    ax.set_ylabel('z (m)')
    ax.set_title(title)

# Show best simulation result time series

Only the total and subsurface selections are rerun and plotted here; the
average-metric selection has no time-series plot. Each selected simulation
is first run with its original parameters. The
minimum-RMSE simulation for each enabled component is then rerun with kmax
rounded to one significant digit and f rounded to one decimal place. The
highlighted curves and reported best parameters and metrics use the rounded
reruns; the displayed metric ranges describe the original selected runs.
Rounding may change the metrics, so the rerun is not necessarily the lowest
RMSE among the original simulations.

In [30]:
if n_sims_tot or n_sims_sbsrf:
    from run_best_krycklan_parameters import run_batch_simulations, run_single_simulation

    project_folder = os.environ.get('PROJECT_FOLDER')
    if not project_folder or not project_folder.strip():
        raise EnvironmentError('PROJECT_FOLDER is not set. Define it in .env or the environment.')

    model_root = Path(project_folder).expanduser() / 'krycklan'
    print(f'Model input root: {model_root}')
else:
    print('No total or subsurface time-series simulations selected.')

Model root: /Users/opa/SpaFHy/krycklan/SpaFHy_RUNS/krycklan
Output folder for netcdf will be: <glue_results_folder>/simulations


## Load observations

In [31]:
if n_sims_tot or n_sims_sbsrf:
    from read_runoff import read_runoff

    catchment_no = 2  # C2 catchment
    runoff_folder = str(model_root / 'obs' / 'DISCHARGE')
    q_obs = read_runoff(catchment_no, runoff_folder)

In [ ]:
catchment = 'krycklan'
if (n_sims_tot and suitable_simulations_tot.empty) or (n_sims_sbsrf and suitable_simulations_sbsrf.empty):
    raise ValueError('No suitable simulations for a selected runoff component; rerun the selection cells.')

print(f'Running simulations for {len(suitable_simulations_tot)} suitable total runoff models...')
results_tot = run_batch_simulations(
    suitable_simulations_tot,
    folder=str(model_root),
    q_obs=q_obs['Q'],
    output_folder=None,
    save_ncf=False,
    catchment=catchment,
    catchment_no=catchment_no,
    soil_type=soil_type,
    skip_existing=True,
    verbose=False,
) if n_sims_tot else {}

print(f'Running simulations for {len(suitable_simulations_sbsrf)} suitable subsurface runoff models...')
results_sbsrf = run_batch_simulations(
    suitable_simulations_sbsrf,
    folder=str(model_root),
    q_obs=q_obs['Q'],
    output_folder=None,
    save_ncf=False,
    catchment=catchment,
    catchment_no=catchment_no,
    soil_type=soil_type,
    skip_existing=True,
    verbose=False,
) if n_sims_sbsrf else {}

print(f'Total simulations completed: {len(results_tot)}')
print(f'Subsurface simulations completed: {len(results_sbsrf)}')

In [ ]:
from scripts.soil_helpers import round_sig


def extract_best_and_ranges(results_dict, suffix):
    """Extract the minimum-RMSE simulation and metric ranges."""
    q_dfs = {}
    params_list = []
    idx_list = []

    for idx, result in results_dict.items():
        if result is not None:
            q_df, params, _ = result
            q_dfs[idx] = q_df
            params_list.append(params)
            idx_list.append(idx)

    if not params_list:
        raise ValueError(f'No completed {suffix} simulations to analyze')
    params_df_all = pd.DataFrame(params_list, index=idx_list)
    best_idx = params_df_all[f'rmse_{suffix}'].idxmin()
    best_q_df = q_dfs[best_idx]
    best_params = params_df_all.loc[best_idx].to_dict()
    ranges = {name: (params_df_all[name].min(), params_df_all[name].max()) for name in base_metrics}
    return best_idx, best_q_df, best_params, ranges, q_dfs


def rerun_best_rounded(best_idx, suitable, suffix):
    row = suitable.loc[best_idx].copy()
    row[f'kmax_{soil_type}'] = round_sig(float(row[f'kmax_{soil_type}']), sig_figs=1)
    row[f'f_{soil_type}'] = round(float(row[f'f_{soil_type}']), 1)
    if row[f'kmax_{soil_type}'] <= row[f'kmin_{soil_type}']:
        raise ValueError(f'Rounded kmax must exceed kmin for the best {suffix} simulation')
    return run_single_simulation(
        row=row,
        folder=str(model_root),
        catchment=catchment,
        catchment_no=catchment_no,
        q_obs=q_obs['Q'],
        soil_type=soil_type,
        create_ncf=False,
    )


best_idx_tot, best_q_df_tot, best_params_tot, ranges_tot, q_dfs_tot = (
    extract_best_and_ranges(results_tot, 'tot') if n_sims_tot else (None, None, None, None, {})
)
best_idx_sbsrf, best_q_df_sbsrf, best_params_sbsrf, ranges_sbsrf, q_dfs_sbsrf = (
    extract_best_and_ranges(results_sbsrf, 'sbsrf') if n_sims_sbsrf else (None, None, None, None, {})
)
if best_idx_tot is not None:
    best_q_df_tot, best_params_tot, _ = rerun_best_rounded(best_idx_tot, suitable_simulations_tot, 'tot')
    print(f"Rounded total runoff rerun: KGE={best_params_tot['kge_tot']:.2f}, RMSE={best_params_tot['rmse_tot']:.2f}")
if best_idx_sbsrf is not None:
    best_q_df_sbsrf, best_params_sbsrf, _ = rerun_best_rounded(best_idx_sbsrf, suitable_simulations_sbsrf, 'sbsrf')
    print(f"Rounded subsurface runoff rerun: KGE={best_params_sbsrf['kge_sbsrf']:.2f}, RMSE={best_params_sbsrf['rmse_sbsrf']:.2f}")

In [ ]:
best_params_tot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('SpaFHy Model - Rounded Best-Selected Simulations vs Observed Runoff', fontsize=14, fontweight='bold')

runoff_types = [
    ('tot', 'q_sim', best_idx_tot, best_params_tot, ranges_tot, q_dfs_tot, best_q_df_tot, axes[0]),
    ('sbsrf', 'q_sim_sbsrf', best_idx_sbsrf, best_params_sbsrf, ranges_sbsrf, q_dfs_sbsrf, best_q_df_sbsrf, axes[1])
]

for runoff_type, q_col, best_idx, best_params, ranges, q_dfs, best_q_df, ax in runoff_types:
    if best_q_df is None:
        ax.set_axis_off()
        ax.set_title(f'{runoff_type.upper()} skipped (0 simulations selected)')
        continue
    overland_flow = best_q_df['q_sim'] - best_q_df['q_sim_sbsrf']
    ax.stackplot(
        best_q_df.index,
        best_q_df['q_sim_sbsrf'],
        overland_flow,
        labels=['Subsurface', 'Overland flow'],
        colors=['tab:orange', 'tab:green'],
        alpha=0.8,
    )

    for idx, q_df in q_dfs.items():
        if idx != best_idx:
            ax.plot(q_df.index, q_df[q_col], color='lightgray', linewidth=1, alpha=0.75, label='_nolegend_')

    ax.plot(best_q_df.index, best_q_df['q_obs'], color='black', linewidth=2, label='Observed', zorder=4)

    kge_val = best_params[f'kge_{runoff_type}']
    kge_min, kge_max = ranges[f'kge_{runoff_type}']
    rmse_val = best_params[f'rmse_{runoff_type}']
    rmse_min, rmse_max = ranges[f'rmse_{runoff_type}']
    mbe_val = best_params[f'mbe_{runoff_type}']
    mbe_min, mbe_max = ranges[f'mbe_{runoff_type}']

    title = (f"{runoff_type.capitalize()} Runoff — rounded rerun (original run ranges) | "
             f"KGE: {kge_val:.2f} ({kge_min:.2f}–{kge_max:.2f}) | "
             f"RMSE: {rmse_val:.1f} ({rmse_min:.1f}–{rmse_max:.1f}) | "
             f"MBE: {mbe_val:.1f} ({mbe_min:.1f}–{mbe_max:.1f})")
    ax.set_title(title, fontsize=11, loc='left')

    ax.set_ylabel('Runoff [mm/d]')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', framealpha=0.9)

if n_sims_tot or n_sims_sbsrf:
    axes[1 if n_sims_sbsrf else 0].set_xlabel('Time')
    plt.tight_layout()
    plt.show()
    print("\nOther original selected simulations are shown in light gray.")
    print("Rounded best-selected simulation components and observed runoff are highlighted.")
else:
    plt.close(fig)
    print('No total or subsurface time-series plots to show.')